In [1]:
import pandas as pd

# create a new dataset for embeddings.
DATA_PATH = "../data/processed/cleaned_patent_data.csv"

df = pd.read_csv(DATA_PATH)

print(df.columns)

Index(['source_url', 'patent_id', 'title', 'title_en', 'abstract',
       'abstract_en', 'description', 'claims', 'filing_date',
       'publication_date', 'country', 'assignee_original', 'assignee_en',
       'ipc_codes', 'cpc_codes', 'claim_count', 'independent_claim_count',
       'backward_citation_count', 'forward_citation_count', 'legal_status',
       'detected_language', 'claims_clean', 'combined_text', 'abstract_length',
       'word_count', 'claims_length_chars', 'claims_word_count',
       'claims_length', 'filing_year', 'year'],
      dtype='object')


In [2]:
df["embedding_text"] = (
    df["title"].fillna("")
    + " "
    + df["abstract_en"].fillna("")
    + " "
    + df["claims_clean"].fillna("")
)

In [3]:
documents = df["embedding_text"].tolist()

print(len(documents))

18352


In [ ]:
#import torch
#from sentence_transformers import SentenceTransformer

#device = "cuda" if torch.cuda.is_available() else "cpu"

#model = SentenceTransformer(
#    "all-MiniLM-L6-v2",
#    device=device
#)

#print("Device:", model.device)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4059.13it/s]


Device: cpu


For Mac

In [5]:
import torch

print("MPS Available:", torch.backends.mps.is_available())
print("MPS Built:", torch.backends.mps.is_built())

MPS Available: True
MPS Built: True


In [7]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="mps"
)

print("Model loaded successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8455.10it/s]


Model loaded successfully!


In [8]:
df["text_length"] = df["embedding_text"].apply(len)

print(df["text_length"].describe())

count     18352.000000
mean       7742.841162
std        8608.694660
min          40.000000
25%        4073.000000
50%        6335.000000
75%        9249.000000
max      519631.000000
Name: text_length, dtype: float64


In [9]:
lengths = [len(x) for x in documents]

import numpy as np

print("Maximum:", np.max(lengths))
print("Average:", np.mean(lengths))
print("Median:", np.median(lengths))

Maximum: 519631
Average: 7742.841161726243
Median: 6335.0


In [10]:
df["embedding_text"] = (
    df["title_en"].fillna("")
    + " "
    + df["abstract_en"].fillna("")
    + " "
    + df["claims_clean"].fillna("").str[:2000]
)

In [11]:
lengths = df["embedding_text"].apply(len)

print("Maximum:", lengths.max())
print("Average:", lengths.mean())
print("Median:", lengths.median())

Maximum: 8983
Average: 2922.9563535309503
Median: 2998.0


In [12]:
test_embeddings = model.encode(
    documents[:100],
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(test_embeddings.shape)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches: 100%|██████████| 7/7 [00:02<00:00,  3.04it/s]

(100, 384)


In [13]:
print(test_embeddings[0])

[-1.97407175e-02  2.09706444e-02  1.69525165e-02 -1.05351731e-01
 -1.10737525e-01 -2.84544174e-02  4.59820442e-02  3.35749090e-02
  5.04732653e-02 -4.13974859e-02  3.72855514e-02  2.59264279e-03
  1.17579311e-01 -6.33503031e-03  6.97375238e-02 -5.66220991e-02
  5.16310781e-02 -4.38175490e-03 -1.80762056e-02  6.48178682e-02
  1.44792171e-02  5.09088933e-02 -1.36549085e-01  4.71757129e-02
 -6.54019788e-02  1.42100975e-02 -7.03681912e-03 -2.45098043e-02
 -4.36542789e-03 -4.58622538e-02  4.66227829e-02  5.01143895e-02
 -3.58782150e-03  7.39403665e-02 -7.53548965e-02 -3.00049800e-02
  1.98055208e-02  4.04942073e-02 -1.05272472e-01  3.03169452e-02
 -5.89741878e-02 -6.71002921e-03 -2.34240610e-02  1.77824055e-03
  6.64544702e-02  1.94689464e-02 -6.82766363e-02  3.30289751e-02
 -2.94754375e-02 -2.77547836e-02 -1.12423934e-01  1.42332725e-02
 -3.98458214e-03  1.80887088e-01  5.36854677e-02 -1.06085315e-02
 -6.24222606e-02  8.18887539e-03 -1.01129949e-01 -1.38299940e-02
 -3.03993151e-02 -5.07699

In [14]:
print(type(test_embeddings))
print(test_embeddings.shape)


<class 'numpy.ndarray'>
(100, 384)


In [15]:
embeddings = embedding_model.encode(
    documents,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches: 100%|██████████| 1147/1147 [05:06<00:00,  3.74it/s]


In [ ]:
#embeddings = model.encode(
#    documents,
#    batch_size=16,
#    show_progress_bar=True,
#    normalize_embeddings=True
#)

#print(embeddings.shape)

Batches: 100%|██████████| 1147/1147 [09:43<00:00,  1.96it/s]


(18352, 384)


In [16]:
embeddings.shape

(18352, 384)

In [17]:
import numpy as np


np.save(
    "../data/processed/patent_embeddings.npy",
    embeddings
)

Verify:-

In [18]:
loaded_embeddings = np.load(
    "../data/processed/patent_embeddings.npy"
)

loaded_embeddings.shape

(18352, 384)